In [3]:
import os
import subprocess
import json
import hashlib
from pathlib import Path
from datetime import datetime

# ===== CONFIGURATION =====

SOURCE_DIR = Path("/Volumes/Green SSD 4/00Workspace_portable/videos")
DEST_DIR = Path("/Volumes/Green SSD 4/00Workspace_portable/mp4videos")
TRACKING_FILE = DEST_DIR / "processed.json"
CRF_VALUE = 18      # Visual quality (18 = near-lossless)
PRESET = "slow"     # Encoding speed/efficiency tradeoff

# ===== SETUP =====
if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"❌ Source directory not found: {SOURCE_DIR}")

DEST_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Source directory: {SOURCE_DIR}")
print(f"✓ Output directory: {DEST_DIR}")
print(f"✓ Tracking file:    {TRACKING_FILE}\n")

# ===== TRACKING FILE MANAGEMENT =====
def load_tracking_registry():
    """Load or initialize tracking registry"""
    if TRACKING_FILE.exists():
        try:
            with open(TRACKING_FILE, 'r', encoding='utf-8') as f:
                data = json.load(f)
                # Migrate legacy format if needed
                if isinstance(data, list):
                    data = {"processed_files": {name: {"timestamp": None} for name in data}, "version": 1}
                print(f"✓ Loaded tracking registry: {len(data.get('processed_files', {}))} processed files")
                return data
        except Exception as e:
            print(f"⚠ Warning: Corrupted tracking file ({e}). Starting fresh registry.")
    return {"processed_files": {}, "version": 2, "created_at": datetime.now().isoformat()}

def partial_hash(filepath, bytes_to_read=1024*1024):
    """Compute MD5 hash of first N bytes for change detection"""
    hasher = hashlib.md5()
    try:
        with open(filepath, 'rb') as f:
            chunk = f.read(bytes_to_read)
            hasher.update(chunk)
    except Exception as e:
        print(f"⚠ Warning: Could not hash {filepath.name}: {e}")
        return None
    return hasher.hexdigest()

def register_processed_file(tracking_data, avi_path, mp4_path):
    """Safely update tracking file after successful conversion (atomic write)"""
    try:
        file_size = avi_path.stat().st_size
        file_hash = partial_hash(avi_path)
        
        record = {
            "original_size_bytes": file_size,
            "original_hash_1mb": file_hash,
            "output_path": str(mp4_path.relative_to(DEST_DIR.parent)),
            "processed_at": datetime.now().isoformat(),
            "ffmpeg_crf": CRF_VALUE,
            "ffmpeg_preset": PRESET
        }
        
        # Update in-memory registry
        tracking_data["processed_files"][avi_path.name] = record
        
        # Atomic write: temp file → fsync → rename
        temp_file = TRACKING_FILE.with_suffix('.json.tmp')
        with open(temp_file, 'w', encoding='utf-8') as f:
            json.dump(tracking_data, f, indent=2, ensure_ascii=False)
            f.flush()
            os.fsync(f.fileno())
        temp_file.replace(TRACKING_FILE)
        
        return True
    except Exception as e:
        print(f"⚠ CRITICAL: Failed to update tracking file for {avi_path.name}: {e}")
        return False

# ===== DISCOVER AVI FILES =====
tracking_data = load_tracking_registry()
registry = tracking_data["processed_files"]

# Collect only .avi files (case-insensitive)
all_files = [p for p in SOURCE_DIR.iterdir() if p.is_file()]
avi_files = [p for p in all_files if p.suffix.lower() == '.avi']
non_avi_videos = [p for p in all_files if p.suffix.lower() in ('.mp4', '.mov', '.mkv', '.wmv') and p.suffix.lower() != '.avi']

print(f"📁 Total files in source:      {len(all_files)}")
print(f"🎥 AVI files found:            {len(avi_files)}")
if non_avi_videos:
    print(f"ℹ️  Non-AVI videos (ignored):  {len(non_avi_videos)} (e.g., {', '.join(f.name for f in non_avi_videos[:3])})")

# Filter: skip already-processed UNCHANGED files
def should_process(avi_path, registry):
    record = registry.get(avi_path.name)
    if not record:
        return True  # Never processed
    
    # Check if file changed since last processing
    current_size = avi_path.stat().st_size
    current_hash = partial_hash(avi_path)
    
    if (record.get("original_size_bytes") == current_size and
        record.get("original_hash_1mb") == current_hash):
        return False  # Unchanged → skip
    
    print(f"🔄 '{avi_path.name}' changed since last processing → will reprocess")
    return True

files_to_process = [f for f in sorted(avi_files) if should_process(f, registry)]
files_skipped = [f for f in avi_files if f not in files_to_process]

print(f"⏭️  Already processed (skipped): {len(files_skipped)}")
print(f"▶️  Files to process now:       {len(files_to_process)}")
if not files_to_process:
    print("\n✅ Nothing to do – all AVI files already processed!")
    exit()

print("\n" + "="*70)

# ===== CONVERSION FUNCTION =====
def convert_single_avi(avi_path, mp4_path):
    """Convert one AVI → MP4 using ffmpeg. Returns (success, message)"""
    try:
        # Detect audio stream
        probe = subprocess.run(
            ["ffmpeg", "-i", str(avi_path), "-hide_banner", "-vn", "-f", "null", "-"],
            capture_output=True, text=True, timeout=10
        )
        has_audio = "Audio:" in probe.stderr
        
        # Build ffmpeg command
        cmd = [
            "ffmpeg", "-y", "-i", str(avi_path),
            "-c:v", "libx264",
            "-crf", str(CRF_VALUE),
            "-preset", PRESET,
            "-vf", "fps=7",          # Preserve 7 FPS
            "-pix_fmt", "yuv420p",   # Max compatibility
        ]
        if has_audio:
            cmd.extend(["-c:a", "aac", "-b:a", "128k"])
        else:
            cmd.append("-an")  # No audio
        cmd.append(str(mp4_path))
        
        # Run conversion
        result = subprocess.run(cmd, capture_output=True, text=True, check=True, timeout=3600)
        return True, f"✓ {avi_path.name} → {mp4_path.name}"
    
    except subprocess.TimeoutExpired:
        return False, f"✗ TIMEOUT {avi_path.name}"
    except subprocess.CalledProcessError as e:
        # Extract last 2 lines of stderr for concise error
        lines = [line.strip() for line in e.stderr.splitlines() if line.strip()][-2:] if e.stderr else []
        err_msg = " | ".join(lines) if lines else str(e)
        return False, f"✗ FFmpeg error {avi_path.name}: {err_msg[:150]}"
    except Exception as e:
        return False, f"✗ Error {avi_path.name}: {str(e)}"

# ===== BATCH PROCESSING WITH PER-FILE TRACKING =====
successful = 0
failed = []

print("Starting conversions...\n")

for idx, avi_file in enumerate(files_to_process, 1):
    mp4_file = DEST_DIR / f"{avi_file.stem}.mp4"
    print(f"[{idx}/{len(files_to_process)}] Processing: {avi_file.name}")
    
    # Convert file
    success, msg = convert_single_avi(avi_file, mp4_file)
    print(f"  {msg}")
    
    if success:
        # ✅ CRITICAL: Update tracking file IMMEDIATELY after success
        if register_processed_file(tracking_data, avi_file, mp4_file):
            successful += 1
            print(f"  💾 Tracking updated for '{avi_file.name}'")
        else:
            failed.append((avi_file.name, "Tracking update failed after successful conversion"))
    else:
        failed.append((avi_file.name, msg))
    print()  # Blank line between files

# ===== FINAL REPORT =====
print("="*70)
print("SUMMARY")
print("="*70)
print(f"✅ Successfully converted & tracked: {successful}/{len(files_to_process)}")
print(f"⏭️  Previously processed (skipped):  {len(files_skipped)}")
if failed:
    print(f"❌ Failed conversions:               {len(failed)}")
    for name, err in failed[:10]:  # Show up to 10 failures
        print(f"   • {name}: {err}")
    if len(failed) > 10:
        print(f"   ... and {len(failed)-10} more failures")

# Space savings report
if successful > 0:
    orig_total = sum(f.stat().st_size for f in files_to_process[:successful])
    new_files = [(DEST_DIR / f"{f.stem}.mp4") for f in files_to_process[:successful]]
    new_total = sum(f.stat().st_size for f in new_files if f.exists())
    
    if new_total > 0:
        savings_pct = (1 - new_total / orig_total) * 100
        print("\n📦 Space efficiency:")
        print(f"   Original AVI size: {orig_total/1024/1024:.1f} MB")
        print(f"   New MP4 size:      {new_total/1024/1024:.1f} MB")
        print(f"   Space saved:       {savings_pct:.1f}%")

print("\n" + "="*70)
print(f"💾 Tracking file: {TRACKING_FILE}")
print(f"   Total files in registry: {len(tracking_data['processed_files'])}")
print("✅ Registry updated AFTER EACH successful conversion (crash-safe)")
print("="*70)

print(1)

✓ Source directory: /Volumes/Green SSD 4/00Workspace_portable/videos
✓ Output directory: /Volumes/Green SSD 4/00Workspace_portable/mp4videos
✓ Tracking file:    /Volumes/Green SSD 4/00Workspace_portable/mp4videos/processed.json

✓ Loaded tracking registry: 48 processed files
📁 Total files in source:      122
🎥 AVI files found:            57
⏭️  Already processed (skipped): 48
▶️  Files to process now:       9

Starting conversions...

[1/9] Processing: 5A.avi
  ✓ 5A.avi → 5A.mp4
  💾 Tracking updated for '5A.avi'

[2/9] Processing: 6A.avi
  ✓ 6A.avi → 6A.mp4
  💾 Tracking updated for '6A.avi'

[3/9] Processing: 6B.avi
  ✓ 6B.avi → 6B.mp4
  💾 Tracking updated for '6B.avi'

[4/9] Processing: 6C.avi
  ✓ 6C.avi → 6C.mp4
  💾 Tracking updated for '6C.avi'

[5/9] Processing: 6D.avi
  ✓ 6D.avi → 6D.mp4
  💾 Tracking updated for '6D.avi'

[6/9] Processing: 7A.avi
  ✓ 7A.avi → 7A.mp4
  💾 Tracking updated for '7A.avi'

[7/9] Processing: 8A.avi
  ✓ 8A.avi → 8A.mp4
  💾 Tracking updated for '8A.avi'

[

In [ ]:
#Google drive

import os
import subprocess
import json
import hashlib
from pathlib import Path
from datetime import datetime

# ===== CONFIGURATION =====

SOURCE_DIR = Path("/content/sample_videos")
DEST_DIR = Path("/content/sample_mp4videos")
TRACKING_FILE = DEST_DIR / "processed.json"
CRF_VALUE = 18      # Visual quality (18 = near-lossless)
PRESET = "slow"     # Encoding speed/efficiency tradeoff

# ===== SETUP =====
if not SOURCE_DIR.exists():
    raise FileNotFoundError(f"❌ Source directory not found: {SOURCE_DIR}")

DEST_DIR.mkdir(parents=True, exist_ok=True)
print(f"✓ Source directory: {SOURCE_DIR}")
print(f"✓ Output directory: {DEST_DIR}")
print(f"✓ Tracking file:    {TRACKING_FILE}\n")

# ===== TRACKING FILE MANAGEMENT =====
def load_tracking_registry():
    """Load or initialize tracking registry"""
    if TRACKING_FILE.exists():
        try:
            with open(TRACKING_FILE, 'r', encoding='utf-8') as f:
                data = json.load(f)
                # Migrate legacy format if needed
                if isinstance(data, list):
                    data = {"processed_files": {name: {"timestamp": None} for name in data}, "version": 1}
                print(f"✓ Loaded tracking registry: {len(data.get('processed_files', {}))} processed files")
                return data
        except Exception as e:
            print(f"⚠ Warning: Corrupted tracking file ({e}). Starting fresh registry.")
    return {"processed_files": {}, "version": 2, "created_at": datetime.now().isoformat()}

def partial_hash(filepath, bytes_to_read=1024*1024):
    """Compute MD5 hash of first N bytes for change detection"""
    hasher = hashlib.md5()
    try:
        with open(filepath, 'rb') as f:
            chunk = f.read(bytes_to_read)
            hasher.update(chunk)
    except Exception as e:
        print(f"⚠ Warning: Could not hash {filepath.name}: {e}")
        return None
    return hasher.hexdigest()

def register_processed_file(tracking_data, avi_path, mp4_path):
    """Safely update tracking file after successful conversion (atomic write)"""
    try:
        file_size = avi_path.stat().st_size
        file_hash = partial_hash(avi_path)
        
        record = {
            "original_size_bytes": file_size,
            "original_hash_1mb": file_hash,
            "output_path": str(mp4_path.relative_to(DEST_DIR.parent)),
            "processed_at": datetime.now().isoformat(),
            "ffmpeg_crf": CRF_VALUE,
            "ffmpeg_preset": PRESET
        }
        
        # Update in-memory registry
        tracking_data["processed_files"][avi_path.name] = record
        
        # Atomic write: temp file → fsync → rename
        temp_file = TRACKING_FILE.with_suffix('.json.tmp')
        with open(temp_file, 'w', encoding='utf-8') as f:
            json.dump(tracking_data, f, indent=2, ensure_ascii=False)
            f.flush()
            os.fsync(f.fileno())
        temp_file.replace(TRACKING_FILE)
        
        return True
    except Exception as e:
        print(f"⚠ CRITICAL: Failed to update tracking file for {avi_path.name}: {e}")
        return False

# ===== DISCOVER AVI FILES =====
tracking_data = load_tracking_registry()
registry = tracking_data["processed_files"]

# Collect only .avi files (case-insensitive)
all_files = [p for p in SOURCE_DIR.iterdir() if p.is_file()]
avi_files = [p for p in all_files if p.suffix.lower() == '.avi']
non_avi_videos = [p for p in all_files if p.suffix.lower() in ('.mp4', '.mov', '.mkv', '.wmv') and p.suffix.lower() != '.avi']

print(f"📁 Total files in source:      {len(all_files)}")
print(f"🎥 AVI files found:            {len(avi_files)}")
if non_avi_videos:
    print(f"ℹ️  Non-AVI videos (ignored):  {len(non_avi_videos)} (e.g., {', '.join(f.name for f in non_avi_videos[:3])})")

# Filter: skip already-processed UNCHANGED files
def should_process(avi_path, registry):
    record = registry.get(avi_path.name)
    if not record:
        return True  # Never processed
    
    # Check if file changed since last processing
    current_size = avi_path.stat().st_size
    current_hash = partial_hash(avi_path)
    
    if (record.get("original_size_bytes") == current_size and
        record.get("original_hash_1mb") == current_hash):
        return False  # Unchanged → skip
    
    print(f"🔄 '{avi_path.name}' changed since last processing → will reprocess")
    return True

files_to_process = [f for f in sorted(avi_files) if should_process(f, registry)]
files_skipped = [f for f in avi_files if f not in files_to_process]

print(f"⏭️  Already processed (skipped): {len(files_skipped)}")
print(f"▶️  Files to process now:       {len(files_to_process)}")
if not files_to_process:
    print("\n✅ Nothing to do – all AVI files already processed!")
    exit()

print("\n" + "="*70)

# ===== CONVERSION FUNCTION =====
def convert_single_avi(avi_path, mp4_path):
    """Convert one AVI → MP4 using ffmpeg. Returns (success, message)"""
    try:
        # Detect audio stream
        probe = subprocess.run(
            ["ffmpeg", "-i", str(avi_path), "-hide_banner", "-vn", "-f", "null", "-"],
            capture_output=True, text=True, timeout=10
        )
        has_audio = "Audio:" in probe.stderr
        
        # Build ffmpeg command
        cmd = [
            "ffmpeg", "-y", "-i", str(avi_path),
            "-c:v", "libx264",
            "-crf", str(CRF_VALUE),
            "-preset", PRESET,
            "-vf", "fps=7",          # Preserve 7 FPS
            "-pix_fmt", "yuv420p",   # Max compatibility
        ]
        if has_audio:
            cmd.extend(["-c:a", "aac", "-b:a", "128k"])
        else:
            cmd.append("-an")  # No audio
        cmd.append(str(mp4_path))
        
        # Run conversion
        result = subprocess.run(cmd, capture_output=True, text=True, check=True, timeout=3600)
        return True, f"✓ {avi_path.name} → {mp4_path.name}"
    
    except subprocess.TimeoutExpired:
        return False, f"✗ TIMEOUT {avi_path.name}"
    except subprocess.CalledProcessError as e:
        # Extract last 2 lines of stderr for concise error
        lines = [line.strip() for line in e.stderr.splitlines() if line.strip()][-2:] if e.stderr else []
        err_msg = " | ".join(lines) if lines else str(e)
        return False, f"✗ FFmpeg error {avi_path.name}: {err_msg[:150]}"
    except Exception as e:
        return False, f"✗ Error {avi_path.name}: {str(e)}"

# ===== BATCH PROCESSING WITH PER-FILE TRACKING =====
successful = 0
failed = []

print("Starting conversions...\n")

for idx, avi_file in enumerate(files_to_process, 1):
    mp4_file = DEST_DIR / f"{avi_file.stem}.mp4"
    print(f"[{idx}/{len(files_to_process)}] Processing: {avi_file.name}")
    
    # Convert file
    success, msg = convert_single_avi(avi_file, mp4_file)
    print(f"  {msg}")
    
    if success:
        # ✅ CRITICAL: Update tracking file IMMEDIATELY after success
        if register_processed_file(tracking_data, avi_file, mp4_file):
            successful += 1
            print(f"  💾 Tracking updated for '{avi_file.name}'")
        else:
            failed.append((avi_file.name, "Tracking update failed after successful conversion"))
    else:
        failed.append((avi_file.name, msg))
    print()  # Blank line between files

# ===== FINAL REPORT =====
print("="*70)
print("SUMMARY")
print("="*70)
print(f"✅ Successfully converted & tracked: {successful}/{len(files_to_process)}")
print(f"⏭️  Previously processed (skipped):  {len(files_skipped)}")
if failed:
    print(f"❌ Failed conversions:               {len(failed)}")
    for name, err in failed[:10]:  # Show up to 10 failures
        print(f"   • {name}: {err}")
    if len(failed) > 10:
        print(f"   ... and {len(failed)-10} more failures")

# Space savings report
if successful > 0:
    orig_total = sum(f.stat().st_size for f in files_to_process[:successful])
    new_files = [(DEST_DIR / f"{f.stem}.mp4") for f in files_to_process[:successful]]
    new_total = sum(f.stat().st_size for f in new_files if f.exists())
    
    if new_total > 0:
        savings_pct = (1 - new_total / orig_total) * 100
        print("\n📦 Space efficiency:")
        print(f"   Original AVI size: {orig_total/1024/1024:.1f} MB")
        print(f"   New MP4 size:      {new_total/1024/1024:.1f} MB")
        print(f"   Space saved:       {savings_pct:.1f}%")

print("\n" + "="*70)
print(f"💾 Tracking file: {TRACKING_FILE}")
print(f"   Total files in registry: {len(tracking_data['processed_files'])}")
print("✅ Registry updated AFTER EACH successful conversion (crash-safe)")
print("="*70)

print(1)